In [ ]:
import os, re, json
import pandas as pd
import kagglehub
from tqdm import tqdm
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document


load_dotenv(override=True)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing")

path = kagglehub.dataset_download("asaniczka/amazon-uk-products-dataset-2023")
csv_file = os.path.join(path, next(f for f in os.listdir(path) if f.lower().endswith(".csv")))
df = pd.read_csv(csv_file)


def parse_price(x):
    if pd.isna(x):
        return None
    s = str(x).replace(",", "").strip()
    s = re.sub(r"[^0-9.]", "", s)
    try:
        return float(s) if s else None
    except:
        return None


def to_float(x):
    if x is None or pd.isna(x):
        return None
    try:
        return float(x)
    except:
        return None


def to_int(x):
    if x is None or pd.isna(x):
        return None
    try:
        return int(float(x))
    except:
        return None


def to_bool(x):
    if x is None or pd.isna(x):
        return None
    try:
        if isinstance(x, bool):
            return x
        s = str(x).strip().lower()
        if s in {"true", "1", "yes"}:
            return True
        if s in {"false", "0", "no"}:
            return False
        return bool(int(float(s)))
    except:
        return None


df["price_num"] = df["price"].apply(parse_price)

needed = ["asin", "title", "categoryName", "imgUrl", "productURL", "price_num"]
df = df.dropna(subset=needed).copy()

targets = {"Women", "Men", "Handmade Clothing, Shoes & Accessories"}
df = df[df["categoryName"].isin(targets)].copy()

df = df[(df["price_num"] > 1) & (df["price_num"] < 2000)].copy()

N = 50000
if len(df) > N:
    df = df.sample(N, random_state=42).copy()

docs = []
for _, row in df.iterrows():
    doc_text = (
        f"Product: {row['title']} | "
        f"Category: {row['categoryName']} | "
        f"Price: {row['price_num']} | "
        f"Stars: {row.get('stars', '')} | "
        f"Reviews: {row.get('reviews', '')} | "
        f"BestSeller: {row.get('isBestSeller', '')} | "
        f"BoughtLastMonth: {row.get('boughtInLastMonth', '')}"
    )

    meta = {
        "asin": row["asin"],
        "title": row["title"],
        "categoryName": row["categoryName"],
        "price": float(row["price_num"]),
        "stars": to_float(row.get("stars")),
        "reviews": to_int(row.get("reviews")),
        "isBestSeller": to_bool(row.get("isBestSeller")),
        "boughtInLastMonth": to_int(row.get("boughtInLastMonth")),
        "imgUrl": row["imgUrl"],
        "productURL": row["productURL"],
    }

    docs.append(Document(page_content=doc_text, metadata=meta))

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

index_dir = "index"
os.makedirs(index_dir, exist_ok=True)

vectorstore = None
BATCH_SIZE = 512

for i in tqdm(range(0, len(docs), BATCH_SIZE), desc="Embedding + FAISS"):
    batch = docs[i:i + BATCH_SIZE]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)

vectorstore.save_local(index_dir)

meta_path = os.path.join(index_dir, "meta.jsonl")
with open(meta_path, "w", encoding="utf-8") as f:
    for d in docs:
        f.write(json.dumps(d.metadata, ensure_ascii=False) + "\n")

print("Saved:", index_dir, "| docs:", len(docs), "| dim:", vectorstore.index.d)
